# My games vs Cheater AI

The games of the main player alone against two computers of the same difficulty and build, by
default **Cheater 3 (Insane)** with the **Economic Focus** build: what is played, how the games go
and what seems to make the difference between the wins and the losses.

- The main player comes from the *Main Player Name* of swarmy's Config, or `MAIN_PLAYER` below.
  The name is compared without the clan tag, `<chezs> Sazed` is `Sazed`.
- The difficulty and the build of the computers are the `lobby_slot_difficulty` and
  `lobby_slot_ai_build` columns of the details, written by s2protocol-rs since the branch
  `snapshot-seconds-and-compression`: 10 is Cheater 3 (Insane), build 5 is Macro, the
  *Economic Focus* of the game lobby.
- The times are game seconds at Faster speed, the in-game clock.
- The structures are counted when they are started, the units when they are born.
- The commands are the orders given, i.e. an Inject Larva commanded twice on the same Hatchery
  counts twice.
- What wins and losses have in common is correlation, not cause: the longer games have more of
  everything, so most comparisons are made at fixed minutes.
- The Cheater 3 (Insane) computers cheat: they see the whole map and they get far more resources
  than their workers could mine. Their income and army are not a target to match, the measures of
  the main player are compared between its own wins and losses. How much more the computers get is
  measured below, per worker.

In [1]:
import json
from pathlib import Path

import plotly.express as px
import polars as pl

import snapshot as s2

pl.Config.set_tbl_width_chars(200)
pl.Config.set_tbl_rows(30)
pl.Config.set_fmt_str_lengths(60)

# The snapshot of the replays, written by s2protocol-rs `write-arrow-ipc` with the AI columns:
# `$S2_IPC_DIR`, or the `ipcs` of the replay directory of the games against the computers.
IPC_DIR = s2.ipc_dir("~/SC2Replays/Sazed/ipcs-notebook")

# The Main Player Name of swarmy's Config, if set.
SWARMY_SETTINGS = Path("~/.local/share/com.swarmy/settings.json").expanduser()
settings = json.loads(SWARMY_SETTINGS.read_text()) if SWARMY_SETTINGS.exists() else {}
MAIN_PLAYER = settings.get("main_player_name") or "Sazed"

# Cheater 3 (Insane), Economic Focus.
AI_DIFFICULTY = 10
AI_BUILD = 5
AI_DIFFICULTIES = {1: "Very Easy", 2: "Easy", 3: "Medium", 4: "Hard", 5: "Harder", 6: "Very Hard",
                   7: "Elite", 8: "Cheater 1 (Vision)", 9: "Cheater 2 (Resources)", 10: "Cheater 3 (Insane)"}
AI_BUILDS = {0: "Any", 1: "Random", 2: "Rush", 3: "Timing", 4: "Power", 5: "Economic Focus", 6: "Air"}

# Win and loss are two categories: the first two slots of the categorical palette, blue and
# orange, they stay apart for the color blind.
RESULT_COLORS = {"Win": "#2a78d6", "Loss": "#eb6834"}
SERIES_COLORS = ["#2a78d6", "#eb6834", "#1baf7a"]
TEMPLATE = "plotly_white"

print(f"Main player {MAIN_PLAYER}, computers {AI_DIFFICULTIES[AI_DIFFICULTY]} {AI_BUILDS[AI_BUILD]}")

Main player Sazed, computers Cheater 3 (Insane) Economic Focus


## The games

One row per game: the main player alone on its team against exactly two computers of the
difficulty and build. The tracker `player_id` of the main player comes from its starting
townhall, the computers have no name in the tracker events.

In [2]:
details = s2.scan(IPC_DIR, "details").filter(pl.col("player_observe") == 0)

computers = details.filter(pl.col("player_control") == 3)
target_games = (
    details.group_by("ext_fs_id")
    .agg(
        pl.len().alias("players"),
        (pl.col("player_control") == 3).sum().alias("computers"),
        pl.col("player_race").filter(pl.col("player_control") == 3).sort().alias("ai_races"),
    )
    .filter((pl.col("players") == 3) & (pl.col("computers") == 2))
    .join(
        computers.group_by("ext_fs_id").agg(
            ((pl.col("lobby_slot_difficulty") == AI_DIFFICULTY) & (pl.col("lobby_slot_ai_build") == AI_BUILD))
            .all()
            .alias("is_target")
        ),
        on="ext_fs_id",
    )
    .filter(pl.col("is_target"))
)

starts = (
    s2.scan(IPC_DIR, "unit_born")
    .filter((pl.col("ext_replay_loop") == 0) & pl.col("unit_type_name").is_in(["Hatchery", "CommandCenter", "Nexus"]))
    .select("ext_fs_id", pl.col("control_player_id").alias("player_id"), "player_name")
)
durations = s2.scan(IPC_DIR, "stats").group_by("ext_fs_id").agg(
    (pl.col("ext_replay_loop").max() / s2.GAME_LOOPS_PER_SECOND).alias("duration")
)

games = (
    details.filter(s2.name_without_clan() == MAIN_PLAYER)
    .join(target_games, on="ext_fs_id", how="semi")
    .join(target_games.select("ext_fs_id", "ai_races"), on="ext_fs_id")
    .join(starts, on=["ext_fs_id", "player_name"])
    .join(durations, on="ext_fs_id")
    .select(
        "ext_fs_id",
        "player_name",
        "player_id",
        pl.col("player_race").alias("race"),
        pl.col("player_result").alias("result"),
        "title",
        "ext_datetime",
        "ext_protocol_version",
        "lobby_random_seed",
        pl.col("ai_races").list.join(" + "),
        "duration",
        pl.col("ext_fs_file_name").alias("file"),
    )
    .filter(pl.col("result").is_in(["Win", "Loss"]))
    .sort("ext_datetime")
    .collect()
)
# The game ids are few, the event scans filter on them.
game_ids = games["ext_fs_id"].to_list()
won = games.select("ext_fs_id", "result").lazy()
# The games that end before 10:00 are all losses, mostly left early: the comparisons of the
# wins and the losses are made with the games that reached 10:00.
FULL_GAME_SECONDS = 600
full_ids = games.filter(pl.col("duration") >= FULL_GAME_SECONDS)["ext_fs_id"].to_list()
print(f"{games.height} games, {games['result'].eq('Win').mean():.1%} won, "
      f"from {games['ext_datetime'].min():%Y-%m-%d} to {games['ext_datetime'].max():%Y-%m-%d}")
print(f"{games.height - len(full_ids)} games ended before 10:00, "
      f"{games.filter(pl.col('duration') < FULL_GAME_SECONDS)['result'].eq('Win').mean():.0%} of them won")
games.head()

1541 games, 32.1% won, from 2024-01-04 to 2026-10-24
238 games ended before 10:00, 0% of them won


ext_fs_id,player_name,player_id,race,result,title,ext_datetime,ext_protocol_version,lobby_random_seed,ai_races,duration,file
u64,str,u8,str,str,str,datetime[ns],u32,u32,str,f64,str
4323,"""&lt;chezs&gt;<sp/>Sazed""",1,"""Zerg""","""Loss""","""Fields of Death""",2024-01-04 19:14:19.996277,91115,841184031,"""Zerg + Zerg""",626.517857,"""/home/seb/SC2Replays/Sazed/urithiru/Fields of Death (425).SC…"
4322,"""&lt;chezs&gt;<sp/>Sazed""",1,"""Zerg""","""Loss""","""Fields of Death""",2024-01-04 19:32:17.727541,91115,1305458871,"""Terran + Zerg""",564.196429,"""/home/seb/SC2Replays/Sazed/urithiru/Fields of Death (426).SC…"
4320,"""&lt;chezs&gt;<sp/>Sazed""",1,"""Zerg""","""Win""","""Fields of Death""",2024-01-04 21:23:44.244003,91115,2969631761,"""Protoss + Protoss""",5939.419643,"""/home/seb/SC2Replays/Sazed/urithiru/Fields of Death (428).SC…"
4314,"""&lt;chezs&gt;<sp/>Sazed""",1,"""Zerg""","""Loss""","""Fields of Death""",2024-01-06 18:59:35.111786,91115,3339933487,"""Protoss + Terran""",865.982143,"""/home/seb/SC2Replays/Sazed/urithiru/Fields of Death (434).SC…"
4312,"""&lt;chezs&gt;<sp/>Sazed""",1,"""Zerg""","""Loss""","""Fields of Death""",2024-01-07 07:41:14.346571,91115,3950441184,"""Protoss + Zerg""",25.089286,"""/home/seb/SC2Replays/Sazed/urithiru/Fields of Death (436).SC…"


### Over time, by map and by the races of the computers

In [3]:
by_month = (
    games.lazy()
    .group_by(pl.col("ext_datetime").dt.truncate("1mo").alias("month"))
    .agg(pl.len().alias("games"), (pl.col("result") == "Win").mean().alias("win_rate"))
    .sort("month")
    .collect()
)
fig = px.line(by_month, x="month", y="win_rate", markers=True, hover_data=["games"], template=TEMPLATE,
              title="Win rate by month", color_discrete_sequence=SERIES_COLORS)
fig.update_yaxes(tickformat=".0%", range=[0, 1])
fig.update_traces(line_width=2, marker_size=8)
fig.show()

by_map, by_races = pl.collect_all([
    games.lazy().group_by("title").agg(pl.len().alias("games"), (pl.col("result") == "Win").mean().round(3).alias("win_rate"),
                                       pl.col("duration").median().truediv(60).round(1).alias("median_minutes"))
    .filter(pl.col("games") >= 10).sort("win_rate", descending=True),
    games.lazy().group_by("ai_races").agg(pl.len().alias("games"), (pl.col("result") == "Win").mean().round(3).alias("win_rate"))
    .sort("games", descending=True),
])
display(by_map)
by_races

title,games,win_rate,median_minutes
str,u32,f64,f64
"""Fields of Death CE""",512,0.381,19.2
"""Fields of Death""",1029,0.291,19.4


ai_races,games,win_rate
str,u32,f64
"""Protoss + Terran""",1102,0.292
"""Protoss + Zerg""",400,0.42
"""Terran + Zerg""",28,0.071
"""Terran + Terran""",6,0.167
"""Protoss + Protoss""",4,0.25
"""Zerg + Zerg""",1,0.0


### How long the games last

The losses that end early are the timings of the computers that were not held, the long games
are won or lost later.

In [4]:
fig = px.histogram(games.with_columns((pl.col("duration") / 60).alias("minutes")), x="minutes", color="result",
                   nbins=60, barmode="overlay", opacity=0.75, template=TEMPLATE, color_discrete_map=RESULT_COLORS,
                   title="Game length, wins and losses")
fig.show()

(
    games.lazy()
    .with_columns(pl.col("duration").truediv(60).cut([10, 15, 20, 30, 45, 60]).alias("minutes"))
    .group_by("minutes")
    .agg(pl.len().alias("games"), (pl.col("result") == "Win").mean().round(3).alias("win_rate"))
    .sort("minutes")
    .collect()
)

minutes,games,win_rate
enum,u32,f64
"""(-inf, 10]""",238,0.0
"""(10, 15]""",402,0.0
"""(15, 20]""",141,0.0
"""(20, 30]""",55,0.0
"""(30, 45]""",69,0.0
"""(45, 60]""",61,0.0
"""(60, inf]""",575,0.859


## Build order and opening

When the main player started each structure, the first time only, and the first army unit
made. The opening is the order of the first Spawning Pool, Extractor and expansion Hatchery
(or the first Barracks, Refinery and Command Center, Gateway, Assimilator and Nexus for the
other races).

In [5]:
born = (
    s2.scan(IPC_DIR, "unit_born")
    .filter(pl.col("ext_fs_id").is_in(game_ids))
    .join(games.lazy().select("ext_fs_id", pl.col("player_id").alias("control_player_id")), on=["ext_fs_id", "control_player_id"], how="semi")
    .filter(~pl.col("unit_type_name").str.starts_with("Beacon"))
    # Burrowing, morphing back, etc register a unit again, a unit is counted once.
    .unique(["ext_fs_id", "unit_tag_index", "unit_tag_recycle"], keep="first")
    .select("ext_fs_id", "unit_type_name", "creator_ability_name", s2.game_seconds())
)

OPENING = {
    "Zerg": {"SpawningPool": "Pool", "Extractor": "Gas", "Hatchery": "Hatch"},
    "Terran": {"Barracks": "Rax", "Refinery": "Gas", "CommandCenter": "CC"},
    "Protoss": {"Gateway": "Gate", "Assimilator": "Gas", "Nexus": "Nexus"},
}
opening_names = {name: label for race in OPENING.values() for name, label in race.items()}
WORKERS_SUPPLY = ["Drone", "SCV", "Probe", "Overlord", "Larva", "Egg", "Broodling", "MULE", "Queen", "Changeling",
                  "CreepTumor", "CreepTumorBurrowed", "CreepTumorQueen", "LocustMP", "LocustMPFlying", "InfestedTerransEgg",
                  "AdeptPhaseShift", "Interceptor", "AutoTurret", "KD8Charge", "ParasiticBombDummy", "RavagerCocoon",
                  "BanelingCocoon", "BroodLordCocoon", "OverlordCocoon", "TransportOverlordCocoon", "LurkerMPEgg",
                  "ForceField", "InvisibleTargetDummy", "BroodlingEscort"]


# Not the workers, supply, cocoons, nor the dummies and indicators of the map.
def is_unit():
    return ~pl.col("unit_type_name").is_in(WORKERS_SUPPLY) & ~pl.col("unit_type_name").str.contains("Dummy|Indicator")


openings = (
    born.filter(pl.col("unit_type_name").is_in(list(opening_names)) & (pl.col("game_seconds") > 0))
    .sort("game_seconds")
    .group_by("ext_fs_id")
    .agg(pl.col("unit_type_name").replace_strict(opening_names).unique(maintain_order=True).head(3).str.join("-").alias("opening"))
)
first_army = (
    # Army units are trained from larva or a production structure, not the initial ones.
    born.filter(pl.col("creator_ability_name").is_not_null() & is_unit())
    .sort("game_seconds")
    .group_by("ext_fs_id")
    .agg(pl.col("unit_type_name").first().alias("first_army_unit"), pl.col("game_seconds").first().alias("first_army_seconds"),
         pl.col("unit_type_name").mode().first().alias("most_made_unit"))
)
structures = ["SpawningPool", "Extractor", "RoachWarren", "BanelingNest", "Lair", "EvolutionChamber",
              "HydraliskDen", "Spire", "InfestationPit", "Hive", "LurkerDenMP", "UltraliskCavern", "GreaterSpire"]
structure_times = (
    born.filter(pl.col("unit_type_name").is_in(structures) & (pl.col("game_seconds") > 0))
    .group_by("ext_fs_id", "unit_type_name")
    .agg(pl.col("game_seconds").min())
    .collect()
    .pivot(on="unit_type_name", index="ext_fs_id", values="game_seconds")
    .lazy()
)
build = (
    games.lazy().select("ext_fs_id", "race", "result")
    .join(openings, on="ext_fs_id", how="left")
    .join(first_army, on="ext_fs_id", how="left")
    .join(structure_times, on="ext_fs_id", how="left")
    .collect()
)
build.group_by("race", "opening").agg(pl.len().alias("games"), (pl.col("result") == "Win").mean().round(3).alias("win_rate")) \
    .filter(pl.col("games") >= 5).sort("games", descending=True)

race,opening,games,win_rate
str,str,u32,f64
"""Zerg""","""Gas-Pool-Hatch""",713,0.367
"""Zerg""","""Gas-Hatch-Pool""",686,0.338
"""Zerg""",null,136,0.0


In [6]:
(
    pl.collect_all([
        build.lazy().group_by("first_army_unit").agg(pl.len().alias("games"), (pl.col("result") == "Win").mean().round(3).alias("win_rate"),
                                                     pl.col("first_army_seconds").median().round(0).alias("median_seconds"))
        .filter(pl.col("games") >= 10).sort("games", descending=True),
        build.lazy().group_by("most_made_unit").agg(pl.len().alias("games"), (pl.col("result") == "Win").mean().round(3).alias("win_rate"))
        .filter(pl.col("games") >= 10).sort("games", descending=True),
    ])
)

[shape: (6, 4)
 ┌─────────────────┬───────┬──────────┬────────────────┐
 │ first_army_unit ┆ games ┆ win_rate ┆ median_seconds │
 │ ---             ┆ ---   ┆ ---      ┆ ---            │
 │ str             ┆ u32   ┆ f64      ┆ f64            │
 ╞═════════════════╪═══════╪══════════╪════════════════╡
 │ Zergling        ┆ 927   ┆ 0.341    ┆ 408.0          │
 │ Hydralisk       ┆ 317   ┆ 0.423    ┆ 437.0          │
 │ null            ┆ 184   ┆ 0.0      ┆ null           │
 │ Infestor        ┆ 69    ┆ 0.449    ┆ 425.0          │
 │ Viper           ┆ 20    ┆ 0.15     ┆ 438.0          │
 │ Corruptor       ┆ 19    ┆ 0.421    ┆ 453.0          │
 └─────────────────┴───────┴──────────┴────────────────┘,
 shape: (4, 3)
 ┌────────────────┬───────┬──────────┐
 │ most_made_unit ┆ games ┆ win_rate │
 │ ---            ┆ ---   ┆ ---      │
 │ str            ┆ u32   ┆ f64      │
 ╞════════════════╪═══════╪══════════╡
 │ Zergling       ┆ 1179  ┆ 0.402    │
 │ null           ┆ 184   ┆ 0.0      │
 │ Hydralisk

### Tech timings, wins against losses

The median time each tech structure was started in the wins and in the losses, and in how many
games it was built. A structure built in most wins and few losses, or earlier in the wins, is
worth a look.

In [7]:
present = [name for name in structures if name in build.columns]
(
    build.lazy()
    .unpivot(index=["ext_fs_id", "result"], on=present, variable_name="structure", value_name="seconds")
    .group_by("structure", "result")
    .agg((pl.col("seconds").is_not_null().mean() * 100).round(0).alias("built_in_pct"),
         pl.col("seconds").median().round(0).alias("median_seconds"))
    .collect()
    .pivot(on="result", index="structure", values=["built_in_pct", "median_seconds"])
    .sort("structure")
)

structure,built_in_pct_Win,built_in_pct_Loss,median_seconds_Win,median_seconds_Loss
str,f64,f64,f64,f64
"""BanelingNest""",92.0,25.0,1562.0,885.0
"""EvolutionChamber""",100.0,82.0,394.0,390.0
"""Extractor""",100.0,87.0,93.0,92.0
"""HydraliskDen""",100.0,83.0,300.0,298.0
"""InfestationPit""",100.0,83.0,311.0,307.0
"""LurkerDenMP""",100.0,82.0,366.0,362.0
"""RoachWarren""",43.0,16.0,931.0,813.0
"""SpawningPool""",100.0,87.0,122.0,122.0
"""Spire""",100.0,80.0,415.0,397.0


## Expansions

The townhalls started by the main player, and how many were alive at each minute: a townhall
counts from when it was started until it died.

In [8]:
TOWNHALLS = ["Hatchery", "Lair", "Hive", "CommandCenter", "OrbitalCommand", "PlanetaryFortress", "Nexus"]
my_units = games.lazy().select("ext_fs_id", pl.col("player_id").alias("control_player_id"))
townhalls = (
    s2.scan(IPC_DIR, "unit_born")
    .filter(pl.col("ext_fs_id").is_in(game_ids) & pl.col("unit_type_name").is_in(["Hatchery", "CommandCenter", "Nexus"]))
    .join(my_units, on=["ext_fs_id", "control_player_id"], how="semi")
    .unique(["ext_fs_id", "unit_tag_index", "unit_tag_recycle"], keep="first")
    .select("ext_fs_id", "unit_tag_index", "unit_tag_recycle", s2.game_seconds().alias("started"))
    .join(
        s2.scan(IPC_DIR, "unit_died").filter(pl.col("ext_fs_id").is_in(game_ids) & pl.col("unit_died_name").is_in(TOWNHALLS))
        .select("ext_fs_id", "unit_tag_index", "unit_tag_recycle", s2.game_seconds().alias("died")),
        on=["ext_fs_id", "unit_tag_index", "unit_tag_recycle"], how="left",
    )
)
MINUTES = [4, 6, 8, 10, 12, 15]
bases = (
    townhalls.group_by("ext_fs_id")
    .agg(
        *[((pl.col("started") <= minute * 60) & (pl.col("died").is_null() | (pl.col("died") > minute * 60))).sum().alias(f"bases_{minute}")
          for minute in MINUTES],
        pl.col("started").filter(pl.col("started") > 0).sort().get(0, null_on_oob=True).alias("expansion_2"),
        pl.col("started").filter(pl.col("started") > 0).sort().get(1, null_on_oob=True).alias("expansion_3"),
        pl.col("started").filter(pl.col("started") > 0).sort().get(2, null_on_oob=True).alias("expansion_4"),
    )
    .join(won, on="ext_fs_id")
    .collect()
)
(
    bases.lazy()
    .group_by("bases_8")
    .agg(pl.len().alias("games"), (pl.col("result") == "Win").mean().round(3).alias("win_rate"))
    .sort("bases_8")
    .collect()
)

bases_8,games,win_rate
u32,u32,f64
1,139,0.0
2,23,0.0
3,13,0.0
4,14,0.143
5,611,0.363
6,379,0.322
7,354,0.412
8,8,0.25


In [9]:
expansions = bases.unpivot(index=["ext_fs_id", "result"], on=["expansion_2", "expansion_3", "expansion_4"],
                           variable_name="expansion", value_name="seconds").drop_nulls("seconds")
fig = px.box(expansions, x="expansion", y="seconds", color="result", template=TEMPLATE, color_discrete_map=RESULT_COLORS,
             title="When the 2nd, 3rd and 4th townhalls were started", points=False)
fig.show()

## Economy and army, minute by minute

From the stats of the tracker events, sampled about every 7 seconds: the workers, the income and
the army value (minerals and vespene of the army alive) of the main player, and of both
computers together. The first sample after each minute is used.

In [10]:
participants = (
    games.lazy().select("ext_fs_id", pl.col("player_id").alias("me"), "result")
)
per_player = (
    s2.scan(IPC_DIR, "stats")
    .filter(pl.col("ext_fs_id").is_in(game_ids))
    .with_columns(s2.game_seconds())
    .with_columns((pl.col("game_seconds") // 60).cast(pl.Int32).alias("minute"))
    .filter(pl.col("minute") <= 20)
    .join(participants, on="ext_fs_id")
    .with_columns(pl.when(pl.col("player_id") == pl.col("me")).then(pl.lit("me")).otherwise(pl.lit("computers")).alias("side"))
    .group_by("ext_fs_id", "result", "side", "player_id", "minute")
    .agg(
        pl.col("workers_active_count").sort_by("game_seconds").first().alias("workers"),
        (pl.col("minerals_collection_rate") + pl.col("vespene_collection_rate")).sort_by("game_seconds").first().alias("income"),
        (pl.col("minerals_used_current_army") + pl.col("vespene_used_current_army")).sort_by("game_seconds").first().alias("army"),
        (pl.col("minerals_lost_army") + pl.col("vespene_lost_army")).sort_by("game_seconds").first().alias("army_lost"),
    )
    .collect()
)
# Both computers together.
per_minute = (
    per_player.lazy()
    .group_by("ext_fs_id", "result", "side", "minute")
    .agg(pl.col("workers", "income", "army", "army_lost").sum())
    .collect()
)
medians = (
    per_minute.lazy()
    .group_by("result", "side", "minute")
    .agg(pl.col("workers", "income", "army").median())
    .with_columns(pl.concat_str("side", pl.lit(", "), "result").alias("series"))
    .sort("minute")
    .collect()
)
# Mine only, the computers cheat.
for measure in ["workers", "income", "army"]:
    fig = px.line(medians.filter(pl.col("side") == "me"), x="minute", y=measure, color="result", template=TEMPLATE,
                  color_discrete_map=RESULT_COLORS, title=f"My {measure}, median of the wins and of the losses", markers=True)
    fig.update_traces(line_width=2, marker_size=8)
    fig.show()

### How much the computers cheat

Measured, not assumed, per phase of the games that lasted until the end of the phase, each
player apart:

- **spent per minute**: the minerals and vespene of everything the player has alive or lost
  (army, economy, technology), from the start to the end of the phase, per minute. It counts the
  resources the computers get without mining them.
- **collection rate per worker**: the income the game reports, divided by the active workers.

The multiplier is the median of a computer divided by the median of the main player, both
computers together are twice that. The resources a computer keeps unspent are not counted.

In [11]:
SPENT = [f"{resource}_{kind}_{category}" for resource in ["minerals", "vespene"] for kind in ["used_current", "lost"]
         for category in ["army", "economy", "technology"]]
PHASES = [(4, 8), (8, 12), (12, 20), (20, 40), (40, 80)]
bounds = sorted({minute for phase in PHASES for minute in phase})
samples = (
    s2.scan(IPC_DIR, "stats")
    .filter(pl.col("ext_fs_id").is_in(game_ids))
    .with_columns(s2.game_seconds())
    .with_columns((pl.col("game_seconds") // 60).cast(pl.Int32).alias("minute"))
    .filter(pl.col("minute").is_in(bounds))
    .join(participants.select("ext_fs_id", "me"), on="ext_fs_id")
    .with_columns(pl.when(pl.col("player_id") == pl.col("me")).then(pl.lit("me")).otherwise(pl.lit("computer")).alias("side"),
                  pl.sum_horizontal(SPENT).alias("spent"),
                  (pl.col("minerals_collection_rate") + pl.col("vespene_collection_rate")).alias("rate"))
    .group_by("ext_fs_id", "player_id", "side", "minute")
    .agg(pl.col("spent", "rate", "workers_active_count").sort_by("game_seconds").first())
    .collect()
)
phases = pl.concat([
    samples.filter(pl.col("minute") == start).join(
        samples.filter(pl.col("minute") == end), on=["ext_fs_id", "player_id", "side"], suffix="_end"
    ).select(
        pl.lit(f"{start:02}:00-{end:02}:00").alias("phase"), "side",
        ((pl.col("spent_end") - pl.col("spent")) / (end - start)).alias("spent_per_minute"),
        (((pl.col("rate") + pl.col("rate_end")) / 2) / ((pl.col("workers_active_count") + pl.col("workers_active_count_end")) / 2).clip(lower_bound=1))
        .alias("rate_per_worker"),
        ((pl.col("workers_active_count") + pl.col("workers_active_count_end")) / 2).alias("workers"),
    )
    for start, end in PHASES
])
cheat = (
    phases.lazy()
    .group_by("phase", "side")
    .agg(pl.len().alias("players"), pl.col("spent_per_minute", "rate_per_worker", "workers").median())
    .collect()
    .pivot(on="side", index="phase", values=["players", "spent_per_minute", "rate_per_worker", "workers"])
    .with_columns(
        (pl.col("spent_per_minute_computer") / pl.col("spent_per_minute_me")).round(2).alias("spent_multiplier"),
        (pl.col("rate_per_worker_computer") / pl.col("rate_per_worker_me")).round(2).alias("rate_per_worker_multiplier"),
    )
    .sort("phase")
)
fig = px.bar(phases.group_by("phase", "side").agg(pl.col("spent_per_minute").median()).sort("phase"), x="phase",
             y="spent_per_minute", color="side", barmode="group", template=TEMPLATE, color_discrete_sequence=SERIES_COLORS,
             title="Resources spent per minute, median, me and each computer")
fig.show()
cheat.with_columns(pl.col(pl.Float64).round(1))

phase,players_me,players_computer,spent_per_minute_me,spent_per_minute_computer,rate_per_worker_me,rate_per_worker_computer,workers_me,workers_computer,spent_multiplier,rate_per_worker_multiplier
str,u32,u32,f64,f64,f64,f64,f64,f64,f64,f64
"""04:00-08:00""",1353,2706,2556.2,2256.2,60.3,73.0,54.5,40.0,0.9,1.2
"""08:00-12:00""",1046,2092,3000.0,2950.0,58.8,86.9,71.0,60.0,1.0,1.5
"""12:00-20:00""",760,1520,1721.9,1750.0,53.9,94.9,55.0,69.5,1.0,1.8
"""20:00-40:00""",657,1314,992.5,1682.5,49.7,73.7,32.5,41.0,1.7,1.5
"""40:00-80:00""",454,908,677.5,1422.1,46.7,11.1,17.0,12.0,2.1,0.2


### The army against both computers

The army value of the main player divided by the army value of both computers, at each minute.
It is what the main player faces, not a target: with their extra income the computers rebuild
faster than any human could, the wins show the ratio that was enough to hold.

In [12]:
ratio = (
    per_minute.lazy()
    .collect()
    .pivot(on="side", index=["ext_fs_id", "result", "minute"], values="army")
    .lazy()
    .with_columns((pl.col("me") / pl.col("computers").clip(lower_bound=1)).alias("army_ratio"))
    .group_by("result", "minute")
    .agg(pl.col("army_ratio").median())
    .sort("minute")
    .collect()
)
fig = px.line(ratio, x="minute", y="army_ratio", color="result", template=TEMPLATE, color_discrete_map=RESULT_COLORS,
              markers=True, title="My army / the army of both computers, median")
fig.update_traces(line_width=2, marker_size=8)
fig.show()

## Actions

The commands of the main player, from both cmd files, a command once (see `s2.casts`), and the
abilities that need attention: Inject Larva, Creep Tumors and Transfuse. They are counted in the
first 10 minutes of the games that reached 10:00, so that the long games do not count more and
the games left early do not count less.

In [13]:
my_names = games.lazy().select("ext_fs_id", "player_name")
my_casts = (
    s2.casts(IPC_DIR)
    .filter(pl.col("ext_fs_id").is_in(game_ids))
    .join(my_names, on=["ext_fs_id", "player_name"], how="semi")
    .with_columns(s2.game_seconds())
)
MACRO_ABILITIES = {"SpawnLarva.Execute": "injects", "Build.CreepTumorQueen": "queen_tumors",
                   "Build.CreepTumor": "tumor_spreads", "Transfusion.Execute": "transfuses"}
actions = (
    my_casts.filter(pl.col("ext_fs_id").is_in(full_ids) & (pl.col("game_seconds") <= FULL_GAME_SECONDS))
    .group_by("ext_fs_id")
    .agg(
        pl.len().alias("commands_first_10"),
        *[(pl.col("ability") == ability).sum().alias(f"{name}_first_10") for ability, name in MACRO_ABILITIES.items()],
    )
    .join(won, on="ext_fs_id")
    .collect()
)
first_10 = ["commands_first_10", *[f"{name}_first_10" for name in MACRO_ABILITIES.values()]]
actions.group_by("result").agg(pl.len().alias("games"), pl.col(first_10).median().round(1))

result,games,commands_first_10,injects_first_10,queen_tumors_first_10,tumor_spreads_first_10,transfuses_first_10
str,u32,f64,f64,f64,f64,f64
"""Win""",274,68.0,24.0,3.0,6.0,0.0
"""Loss""",393,66.0,22.0,3.0,6.0,1.0


## Where the losses happen

At what minute, from 6:00, the army value of the main player first drops below half of its
usual army at that minute (the median of the wins), in the wins and in the losses, and the computer units that killed the most units of
the main player in each.

In [14]:
# The reference is the usual army of the wins at the same minute, not the cheating computers.
win_army = (
    per_minute.lazy().filter((pl.col("side") == "me") & (pl.col("result") == "Win"))
    .group_by("minute").agg(pl.col("army").median().alias("win_median_army"))
)
behind = (
    per_minute.lazy()
    .filter(pl.col("side") == "me")
    .join(win_army, on="minute")
    .filter((pl.col("minute") >= 6) & (pl.col("army") < 0.5 * pl.col("win_median_army")))
    .group_by("ext_fs_id", "result")
    .agg(pl.col("minute").min().alias("first_minute_behind"))
    .collect()
)
fig = px.histogram(behind, x="first_minute_behind", color="result", barmode="group", template=TEMPLATE,
                   color_discrete_map=RESULT_COLORS, title="First minute with less than half the usual army of the wins")
fig.show()
behind.group_by("result").agg(pl.len().alias("games_behind_at_some_point"), pl.col("first_minute_behind").median())

result,games_behind_at_some_point,first_minute_behind
str,u32,f64
"""Loss""",102,13.0
"""Win""",15,8.0


In [15]:
my_unit_tags = (
    s2.scan(IPC_DIR, "unit_born")
    .filter(pl.col("ext_fs_id").is_in(game_ids))
    .join(my_units, on=["ext_fs_id", "control_player_id"], how="semi")
    .select("ext_fs_id", "unit_tag_index", "unit_tag_recycle")
    .unique()
)
killed_by = (
    s2.scan(IPC_DIR, "unit_died")
    .filter(pl.col("ext_fs_id").is_in(game_ids) & pl.col("unit_killer_name").is_not_null())
    .join(my_unit_tags, on=["ext_fs_id", "unit_tag_index", "unit_tag_recycle"], how="semi")
    .join(games.lazy().select("ext_fs_id", "player_id", "result"), on="ext_fs_id")
    .filter(pl.col("killer_player_id") != pl.col("player_id"))
    .filter(~pl.col("unit_died_name").is_in(["Larva", "Egg", "Broodling", "CreepTumorBurrowed", "Changeling"]))
    .group_by("result", "unit_killer_name")
    .agg(pl.len().alias("my_units_killed"))
    .with_columns((pl.col("my_units_killed") / pl.col("my_units_killed").sum().over("result") * 100).round(1).alias("pct_of_my_losses"))
    .collect()
    .pivot(on="result", index="unit_killer_name", values="pct_of_my_losses")
    .with_columns((pl.col("Loss") - pl.col("Win")).round(1).alias("more_in_losses"))
    .sort("more_in_losses", descending=True, nulls_last=True)
)
killed_by.head(15)

unit_killer_name,Win,Loss,more_in_losses
str,f64,f64,f64
"""Stalker""",8.6,11.4,2.8
"""SiegeTankSieged""",5.9,7.9,2.0
"""Marauder""",4.5,6.2,1.7
"""VikingFighter""",1.5,2.6,1.1
"""Immortal""",2.9,3.8,0.9
"""Roach""",2.7,3.5,0.8
"""Colossus""",3.8,4.6,0.8
"""Thor""",0.7,1.3,0.6
"""Hydralisk""",0.9,1.4,0.5


## What seems to work

Every measure of a game that reached 10:00, split in four equal groups of games from its lowest
to its highest value (quartiles), with the win rate of each group. A timing in seconds is better
low, a count is better high. The measures whose win rate changes the most
from the lowest to the highest group tell what goes with winning: the workers, the army and the
bases at fixed minutes, the timings of the opening and the actions.

In [16]:
army_at = per_minute.filter(pl.col("side") == "me").pivot(on="minute", index="ext_fs_id", values=["workers", "army", "income"])
keep_minutes = [f"{measure}_{minute}" for measure in ["workers", "army", "income"] for minute in [4, 6, 8, 10]]
features = (
    games.lazy().filter(pl.col("ext_fs_id").is_in(full_ids)).select("ext_fs_id", "result")
    .join(build.lazy().select("ext_fs_id", "first_army_seconds", *[name for name in ["SpawningPool", "Extractor", "Lair", "RoachWarren", "EvolutionChamber"] if name in build.columns]), on="ext_fs_id", how="left")
    .join(bases.lazy().select("ext_fs_id", "expansion_2", "expansion_3", "expansion_4", "bases_6", "bases_8", "bases_10"), on="ext_fs_id", how="left")
    .join(actions.lazy().select("ext_fs_id", *first_10), on="ext_fs_id", how="left")
    .join(army_at.lazy().select("ext_fs_id", *[name for name in keep_minutes if name in army_at.columns]), on="ext_fs_id", how="left")
    .collect()
)
measures = [name for name in features.columns if name not in ("ext_fs_id", "result")]
quartiles = pl.concat(
    [
        features.lazy()
        .filter(pl.col(measure).is_not_null())
        .with_columns(pl.col(measure).qcut(4, labels=["Q1 lowest", "Q2", "Q3", "Q4 highest"], allow_duplicates=True).alias("quartile"))
        .group_by("quartile")
        .agg(pl.len().alias("games"), (pl.col("result") == "Win").mean().alias("win_rate"), pl.col(measure).median().alias("median"))
        .with_columns(pl.lit(measure).alias("measure"))
        for measure in measures
    ],
    how="vertical_relaxed",
).collect()
effect = (
    quartiles.pivot(on="quartile", index="measure", values="win_rate")
    .select("measure", "Q1 lowest", "Q2", "Q3", "Q4 highest")
    .with_columns((pl.col("Q4 highest") - pl.col("Q1 lowest")).alias("highest_minus_lowest"))
    .sort(pl.col("highest_minus_lowest").abs(), descending=True)
)
effect.with_columns(pl.exclude("measure").round(3))

measure,Q1 lowest,Q2,Q3,Q4 highest,highest_minus_lowest
str,f64,f64,f64,f64,f64
"""RoachWarren""",0.442,0.505,0.589,0.684,0.242
"""injects_first_10""",0.283,0.414,0.461,0.511,0.228
"""workers_6""",0.273,0.352,0.413,0.498,0.226
"""workers_8""",0.293,0.327,0.405,0.498,0.206
"""income_6""",0.264,0.412,0.373,0.469,0.205
"""commands_first_10""",0.261,0.461,0.469,0.464,0.202
"""income_8""",0.297,0.296,0.439,0.486,0.189
"""income_10""",0.282,0.395,0.37,0.469,0.187
"""EvolutionChamber""",0.298,0.373,0.377,0.471,0.173


In [17]:
top = effect.head(6)["measure"].to_list()
fig = px.bar(quartiles.filter(pl.col("measure").is_in(top)).sort("quartile"), x="quartile", y="win_rate", facet_col="measure",
             facet_col_wrap=3, hover_data=["games", "median"], template=TEMPLATE, color_discrete_sequence=SERIES_COLORS,
             title="Win rate by quartile of the measures that change it the most")
fig.update_yaxes(tickformat=".0%", range=[0, 1])
fig.for_each_annotation(lambda annotation: annotation.update(text=annotation.text.split("=")[-1]))
fig.show()

## The random seed of the game

Every game has a random seed (`lobby_random_seed`), the computers decide with it among the
choices of their build, so it may explain why the same build plays differently. It can only be
seen through what the computers do: if the seed mattered for the result, the win rate would
change between groups of seeds, and the same seed would give the same openings of the
computers. The first army unit and the first expansion of each computer are their opening.

In [18]:
seed_groups = (
    games.lazy()
    .with_columns(pl.col("lobby_random_seed").qcut(10, allow_duplicates=True).alias("seed_decile"))
    .group_by("seed_decile")
    .agg(pl.len().alias("games"), (pl.col("result") == "Win").mean().round(3).alias("win_rate"),
         pl.col("lobby_random_seed").min().alias("from_seed"))
    .sort("from_seed")
    .collect()
)
print(f"{games['lobby_random_seed'].n_unique()} different seeds in {games.height} games")
# With about 150 games per group, the win rate of a group varies by about 4 points by chance.
win_rate = games["result"].eq("Win").mean()
print(f"Standard error of a group: {(win_rate * (1 - win_rate) / (games.height / 10)) ** 0.5:.3f}")
seed_groups

1524 different seeds in 1541 games
Standard error of a group: 0.038


seed_decile,games,win_rate,from_seed
cat,u32,f64,u32
"""(-inf, 369752386]""",155,0.297,2711548
"""(369752386, 783525272]""",154,0.318,370535729
"""(783525272, 1272332751]""",154,0.292,784525872
"""(1272332751, 1663067398]""",154,0.331,1275560514
"""(1663067398, 2140216245]""",154,0.39,1663567821
"""(2140216245, 2585209162]""",154,0.338,2144589564
"""(2585209162, 2985612802]""",154,0.357,2589606474
"""(2985612802, 3403239024]""",154,0.299,2986072643
"""(3403239024, 3843742894]""",154,0.344,3405524009


In [19]:
ai_born = (
    s2.scan(IPC_DIR, "unit_born")
    .filter(pl.col("ext_fs_id").is_in(game_ids) & (pl.col("ext_replay_loop") > 0) & pl.col("control_player_id").is_in([1, 2, 3, 4]))
    .join(games.lazy().select("ext_fs_id", "player_id"), on="ext_fs_id")
    .filter(pl.col("control_player_id") != pl.col("player_id"))
    .unique(["ext_fs_id", "unit_tag_index", "unit_tag_recycle"], keep="first")
    .select("ext_fs_id", "control_player_id", "unit_type_name", "creator_ability_name", s2.game_seconds())
)
# The units are trained, the townhalls are started without a creator ability.
ai_units = ai_born.filter(pl.col("creator_ability_name").is_not_null())
ai_openings = (
    ai_units.filter(is_unit())
    .sort("game_seconds")
    .group_by("ext_fs_id", "control_player_id")
    .agg(pl.col("unit_type_name").first().alias("ai_first_army_unit"), pl.col("game_seconds").first().round(0).alias("ai_first_army_seconds"))
    .join(
        ai_born.filter(pl.col("unit_type_name").is_in(["CommandCenter", "Nexus", "Hatchery"]))
        .group_by("ext_fs_id", "control_player_id").agg(pl.col("game_seconds").min().round(0).alias("ai_expansion_seconds")),
        on=["ext_fs_id", "control_player_id"], how="left",
    )
    .join(won, on="ext_fs_id")
    .collect()
)
(
    ai_openings.lazy()
    .group_by("ai_first_army_unit")
    .agg(pl.len().alias("computers"), (pl.col("result") == "Win").mean().round(3).alias("my_win_rate"),
         pl.col("ai_first_army_seconds").median(), pl.col("ai_expansion_seconds").median())
    .filter(pl.col("computers") >= 10)
    .sort("computers", descending=True)
    .collect()
)

ai_first_army_unit,computers,my_win_rate,ai_first_army_seconds,ai_expansion_seconds
str,u32,f64,f64,f64
"""Zealot""",1372,0.359,122.0,228.0
"""Marine""",1026,0.318,108.0,390.0
"""Zergling""",405,0.415,125.0,139.0


### Early or late expansions of the computers

The Economic Focus computers expand early; when one of them did not, it attacked instead. The win
rate by when the computers took their first expansion, the earliest of the two.

In [20]:
(
    ai_openings.lazy()
    .group_by("ext_fs_id", "result")
    .agg(pl.col("ai_expansion_seconds").min().alias("first_ai_expansion"), pl.col("ai_first_army_seconds").min().alias("first_ai_army"))
    .with_columns(pl.col("first_ai_expansion").cut([60, 90, 120, 180, 240]).alias("first_ai_expansion_seconds"))
    .group_by("first_ai_expansion_seconds")
    .agg(pl.len().alias("games"), (pl.col("result") == "Win").mean().round(3).alias("win_rate"), pl.col("first_ai_army").median())
    .sort("first_ai_expansion_seconds")
    .collect()
)

first_ai_expansion_seconds,games,win_rate,first_ai_army
enum,u32,f64,f64
null,15,0.0,110.0
"""(90, 120]""",69,0.464,122.0
"""(120, 180]""",695,0.331,108.0
"""(180, 240]""",318,0.352,112.0
"""(240, inf]""",306,0.392,107.0


## The best and the worst games

The wins with the most army against both computers at 10:00 and the losses with the least, to
open in swarmy (Replay Detail, by the file) and watch.

In [21]:
at_10 = (
    per_minute.filter(pl.col("minute") == 10)
    .pivot(on="side", index=["ext_fs_id", "result"], values="army")
    .with_columns((pl.col("me") / pl.col("computers").clip(lower_bound=1)).round(2).alias("army_ratio_10"))
    .join(games.select("ext_fs_id", "title", "ext_datetime", (pl.col("duration") / 60).round(1).alias("minutes"), "file"), on="ext_fs_id")
)
display(at_10.filter(pl.col("result") == "Win").sort("army_ratio_10", descending=True).head(5))
at_10.filter(pl.col("result") == "Loss").sort("army_ratio_10").head(5)

ext_fs_id,result,computers,me,army_ratio_10,title,ext_datetime,minutes,file
u64,str,i32,i32,f64,str,datetime[ns],f64,str
5264,"""Win""",8225,8350,1.02,"""Fields of Death""",2025-03-13 18:31:02.132513,68.7,"""/home/seb/SC2Replays/Sazed/urithiru/Fields of Death (1118).S…"
5275,"""Win""",8600,8100,0.94,"""Fields of Death""",2025-03-19 19:41:45.490142,93.9,"""/home/seb/SC2Replays/Sazed/urithiru/Fields of Death (1129).S…"
5901,"""Win""",10275,9525,0.93,"""Fields of Death""",2025-10-30 17:10:33.231259,84.5,"""/home/seb/SC2Replays/Sazed/urithiru/Fields of Death (1499).S…"
75,"""Win""",7800,7175,0.92,"""Fields of Death CE""",2026-05-23 15:22:06.574721,91.0,"""/home/seb/SC2Replays/Sazed/wormgat/Multiplayer/Fields of Dea…"
5899,"""Win""",9325,8550,0.92,"""Fields of Death""",2025-10-31 20:29:09.775205,112.7,"""/home/seb/SC2Replays/Sazed/urithiru/Fields of Death (1501).S…"


ext_fs_id,result,computers,me,army_ratio_10,title,ext_datetime,minutes,file
u64,str,i32,i32,f64,str,datetime[ns],f64,str
4722,"""Loss""",17600,3325,0.19,"""Fields of Death""",2024-07-20 20:08:43.418257,61.2,"""/home/seb/SC2Replays/Sazed/urithiru/Fields of Death (803).SC…"
4309,"""Loss""",17300,3850,0.22,"""Fields of Death""",2024-01-07 11:16:39.109075,61.0,"""/home/seb/SC2Replays/Sazed/urithiru/Fields of Death (439).SC…"
4574,"""Loss""",17375,3950,0.23,"""Fields of Death""",2024-02-19 16:49:38.191427,31.2,"""/home/seb/SC2Replays/Sazed/urithiru/Fields of Death (522).SC…"
4256,"""Loss""",19225,5000,0.26,"""Fields of Death""",2024-02-03 19:19:36.298221,11.1,"""/home/seb/SC2Replays/Sazed/urithiru/Fields of Death (486).SC…"
4494,"""Loss""",8950,2300,0.26,"""Fields of Death""",2024-04-01 12:36:13.699947,10.0,"""/home/seb/SC2Replays/Sazed/urithiru/Fields of Death (601).SC…"
